<a href="https://colab.research.google.com/github/lotlotss/F07_Roslyn_Contrevida_IT_Elect1/blob/main/Student_Prediction_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import joblib
import numpy as np
from sklearn.tree import DecisionTreeClassifier

# Sample training data
study_hours = [[1], [2], [3], [4], [5], [6], [7], [8]]

results = [
    "Fail", "Fail", "Fail", "Pass",
    "Pass", "Pass", "Pass", "Pass"
]

# Train the model
model = DecisionTreeClassifier(random_state=42)
model.fit(study_hours, results)

# Save and load the model
joblib.dump(model, "student_pass_model.joblib")
print("Model saved successfully!")

loaded_model = joblib.load("student_pass_model.joblib")
print("Model loaded successfully!")

# Make three predictions
test_hours = [2, 5, 7]

for hours in test_hours:
    input_data = np.array([[hours]])
    print("Input shape:", input_data.shape)

    prediction = loaded_model.predict(input_data)

    print(f"Study Hours: {hours} - Prediction: {prediction[0]}")

Model saved successfully!
Model loaded successfully!
Input shape: (1, 1)
Study Hours: 2 - Prediction: Fail
Input shape: (1, 1)
Study Hours: 5 - Prediction: Pass
Input shape: (1, 1)
Study Hours: 7 - Prediction: Pass


In [ ]:

%pip install flask joblib scikit-learn

In [ ]:
import joblib
import os

print("Model file exists:", os.path.exists("student_pass_model.joblib"))

Model file exists: True


In [ ]:
import joblib
import numpy as np

loaded_model = joblib.load("student_pass_model.joblib")

test_hours = [2, 5, 7]

for hours in test_hours:
    prediction = loaded_model.predict(np.array([[hours]]))
    print(f"Study Hours: {hours} - Prediction: {prediction[0]}")

Study Hours: 2 - Prediction: Fail
Study Hours: 5 - Prediction: Pass
Study Hours: 7 - Prediction: Pass


In [ ]:
from google.colab import files

files.download("student_pass_model.joblib")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from flask import Flask, request, jsonify
import threading
import requests

app = Flask("student_prediction_api")

loaded_model = joblib.load("student_pass_model.joblib")

@app.route("/predict", methods=["POST"])
def predict():
    data = request.get_json(silent=True)

    if not isinstance(data, dict):
        return jsonify({"error": "Please send JSON data."}), 400

    hours = data.get("study_hours")
    attendance = data.get("attendance_percentage")

    if isinstance(hours, bool) or not isinstance(hours, (int, float)):
        return jsonify({"error": "Study hours must be a number."}), 400

    if not 0 <= hours <= 24:
        return jsonify({"error": "Study hours must be between 0 and 24."}), 400

    if isinstance(attendance, bool) or not isinstance(attendance, (int, float)):
        return jsonify({"error": "Attendance must be a number."}), 400

    if not 0 <= attendance <= 100:
        return jsonify({"error": "Attendance must be between 0 and 100."}), 400

    prediction = loaded_model.predict([[hours]])[0]

    return jsonify({
        "study_hours": hours,
        "attendance_percentage": attendance,
        "prediction": str(prediction)
    })

threading.Thread(
    target=lambda: app.run(
        host="127.0.0.1",
        port=5000,
        debug=False,
        use_reloader=False
    ),
    daemon=True
).start()

print("Flask API started!")

Flask API started!
 * Serving Flask app 'student_prediction_api'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000


In [ ]:
response = requests.post(
    "http://127.0.0.1:5000/predict",
    json={
        "study_hours": 5,
        "attendance_percentage": 90
    }
)

print("Status Code:", response.status_code)
print("Response:", response.json())

INFO:werkzeug:127.0.0.1 - - [10/Oct/2026 22:55:44] "POST /predict HTTP/1.1" 200 -


Status Code: 200
Response: {'attendance_percentage': 90, 'prediction': 'Pass', 'study_hours': 5}


In [ ]:
response = requests.post(
    "http://127.0.0.1:5000/predict",
    json={
        "study_hours": 5,
        "attendance_percentage": 120
    }
)

print("Status Code:", response.status_code)
print("Response:", response.json())

INFO:werkzeug:127.0.0.1 - - [10/Oct/2026 22:56:28] "POST /predict HTTP/1.1" 400 -


Status Code: 400
Response: {'error': 'Attendance must be between 0 and 100.'}


In [ ]:
%pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 94.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 68.7 MB/s eta 0:00:00


In [ ]:
%%writefile app.py
import streamlit as st
import joblib

model = joblib.load("student_pass_model.joblib")

st.title("Student Pass Prediction System")
st.write("Predict a student's result based on study hours.")

student_name = st.text_input("Student Name")
study_hours = st.number_input(
    "Study Hours",
    min_value=0.0,
    max_value=24.0,
    value=3.0,
    step=0.5
)

if st.button("Predict Result"):
    if not student_name.strip():
        st.warning("Please enter the student's name.")
    else:
        prediction = model.predict([[study_hours]])[0]

        if prediction == "Pass":
            st.success(f"{student_name} is predicted to PASS!")
        else:
            st.error(f"{student_name} is predicted to FAIL.")

        st.write("### How was this prediction generated?")
        st.write(
            f"The trained Decision Tree model used "
            f"{study_hours} study hours to predict the result. "
            "The model was trained using sample study-hour data "
            "and their corresponding Pass/Fail results."
        )

Writing app.py


In [ ]:
%pip install pyngrok

In [ ]:
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 &>/content/streamlit.log &

In [ ]:
!cat /content/streamlit.log



2026-10-10 22:59:36.772 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.48.114.102:8501



In [ ]:
!pkill -f streamlit

In [ ]:
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧
added 22 packages in 3s
⠧
⠧3 packages are looking for funding
⠧  run `npm fund` for details
⠧

In [ ]:
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 > /content/streamlit.log 2>&1 &

In [ ]:
!lt --port 8501

your url is: https://salty-rules-rule.loca.lt
^C


In [ ]:
!cat /content/streamlit.log



2026-10-10 23:06:17.654 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.48.114.102:8501

  Stopping...


In [ ]:
!curl -s https://api.ipify.org

34.48.114.102

In [ ]:
!curl -I http://127.0.0.1:8501

curl: (7) Failed to connect to 127.0.0.1 port 8501 after 0 ms: Couldn't connect to server


In [ ]:
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 > /content/streamlit.log 2>&1 &

In [ ]:
!sleep 5
!curl -I http://127.0.0.1:8501

HTTP/1.1 200 OK
date: Sat, 10 Oct 2026 23:37:09 GMT
server: uvicorn
content-type: text/html; charset=utf-8
accept-ranges: bytes
content-length: 6463
last-modified: Sat, 10 Oct 2026 22:57:21 GMT
etag: "80c2a8cb1ce1a3fb23f57817e26ffe0a"
cache-control: no-cache



In [ ]:
!lt --port 8501

your url is: https://beige-rings-fetch.loca.lt
